In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

# import FCI code
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.cit import fisherz
from causallearn.utils.GraphUtils import GraphUtils

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
nij_root = project_root / "data" / "processed"
#test it works
nij_root

output_dir = project_root/"results"/"NIJ"/"graphs_fci"
output_dir.mkdir(parents=True,exist_ok=True)

In [2]:
#scoring utility function given two adjacency matricies
def score_graph(W_est, W_true, labels_est, labels_true):
    #number of variables
    p = W_est.shape[0]

    #node orderings
    labels_est = list(labels_est)
    labels_true = list(labels_true)
    common = sorted(set(labels_est) & set(labels_true))

    #map ids
    idx_est = [labels_est.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_est_aligned = np.asarray(W_est)[np.ix_(idx_est, idx_est)]
    W_true_aligned = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    est = (W_est!=0).astype(int)
    true = (W_true !=0).astype(int)

    #true positive, false positive, false negative, true negative
    tp = np.sum((est==1) & (true==1))
    fp = np.sum((est==1) & (true == 0))
    fn = np.sum((est==0) & (true == 1))
    tn = np.sum((est==0) & (true == 0))

    #structural hamming distance, true positive rate, false positive rate
    shd = fp + fn
    tpr = tp/(tp+fn) if (tp+fn) > 0 else np.nan
    fpr = fp/(fp+tn) if (fp + tn)>0 else np.nan
    fdr = fp/(tp+fp) if (tp+fp)>0 else np.nan

    #store scores in dictionary format for each 'experiment'
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), SHD=int(shd), TPR = tpr, FPR = fpr, FDR= fdr)

In [3]:
def is_dir(W,i,j):
    # an edge is oriented i -> j if j has arrowhead (enc as 1) and i doesn't (anything but 1)
    return (W[i, j] != 1) and (W[j, i] == 1)

#pdag encoding -1 -> arrowtail, 1 -> arrowhead, 2 -> circle mark
def score_pdag(W_pdag, W_true, labels_pdag, labels_true):
    W_pdag = np.asarray(W_pdag)
    W_true = np.asarray(W_true)

    #skeleton graphs ignore directionality, score presence of edge
    S_est = ((W_pdag != 0) | (W_pdag.T != 0)).astype(int)
    S_true = ((W_true != 0) | (W_true.T != 0)).astype(int)

    s_metrics = score_graph(S_est,S_true, labels_pdag, labels_true) #(is it right that we should ignore false pos and false neg?)

    #orientation metric, score directed edges
    #+rearrange adjacency matrix if required
    
    #node orderings
    labels_pdag = list(labels_pdag)
    labels_true = list(labels_true)
    common = sorted(set(labels_pdag) & set(labels_true))

    #map ids
    idx_est = [labels_pdag.index(v) for v in common]
    idx_true = [labels_true.index(v) for v in common]

    W_pdag = np.asarray(W_pdag)[np.ix_(idx_est, idx_est)]
    W_true = np.asarray(W_true)[np.ix_(idx_true, idx_true)]

    #number of variables
    p = W_true.shape[0]
    tp_dir = fp_dir=fn_dir = tn_dir=0

    for i in range(p):
        for j in range(p):
            if i ==j:
                #print("i was equal j")
                continue

            true_ij = W_true[i,j]
            true_ji = W_true[j,i]
            est_ij = W_pdag[i,j]
            est_ji= W_pdag[j,i]

            #consider only pairs that are adjacent in ground truth DAG
            if(true_ij==0) and (true_ji==0):
                #print("wasnt adjacent in gt")
                continue

            # set correct direction according to the ground truth
            if true_ij == 1:
                true_dir = (i,j)
            else:
                true_dir = (j,i)

            #score direction according to the directed edge in the PDAG
            if is_dir(W_pdag, i, j):
                est_dir = (i, j)
            elif is_dir(W_pdag, j, i):
                est_dir = (j, i)
            elif (W_pdag[i, j] != 0) or (W_pdag[j, i] != 0):
                est_dir = None   # edge not oriented
            else:
                est_dir = None   # no edge

            if est_dir is None:
                # adjacency handled by skeleton metrics
                fn_dir += 1   # methodological choice: can choose to count lack of orientation as FN..
            elif est_dir == true_dir:
                tp_dir += 1
            else:
                fp_dir += 1

    shd_dir = fp_dir + fn_dir
    tpr_dir = tp_dir/(tp_dir+fn_dir) if (tp_dir+fn_dir) > 0 else np.nan
    fpr_dir = fp_dir/(fp_dir+tn_dir) if (fp_dir+tn_dir) > 0 else np.nan
    fdr_dir = fp_dir/(tp_dir+fp_dir) if (tp_dir+fp_dir) > 0 else np.nan

    orient_metrics = dict(
        TP=int(tp_dir), FP=int(fp_dir), FN=int(fn_dir), TN=int(tn_dir),
        SHD=int(shd_dir), TPR=tpr_dir, FPR=fpr_dir, FDR=fdr_dir
    )

    #return metric dictionaries, prefix with skeleton or orientation
    return {
        **{f"skel_{k}": v for k, v in s_metrics.items()},
        **{f"orient_{k}": v for k, v in orient_metrics.items()},
    }


In [11]:
#path to dataset
csv_path = nij_root/"NIJ_clean.csv"

#load into df and sanity check form
df = pd.read_csv(csv_path)
df.head(), df.shape

(   ID Gender   Race Age_at_Release  Residence_PUMA  Gang_Affiliated  \
 0   1      M  BLACK          43-47              16            False   
 1   2      M  BLACK          33-37              16            False   
 2   3      M  BLACK    48 or older              24            False   
 3   4      M  WHITE          38-42              16            False   
 4   5      M  WHITE          33-37              16            False   
 
    Supervision_Risk_Score_First Supervision_Level_First  \
 0                           3.0                Standard   
 1                           6.0             Specialized   
 2                           7.0                    High   
 3                           7.0                    High   
 4                           4.0             Specialized   
 
          Education_Level Dependents  ... Recidivism_Arrest_Year2  \
 0  At least some college  3 or more  ...                   False   
 1   Less than HS diploma          1  ...                   False 

In [12]:
df_enc = pd.get_dummies(df)
data = df_enc.to_numpy(dtype=float)
g, _ = fci(data, fisherz, alpha=0.05) #causal graph object

  0%|          | 0/149 [00:00<?, ?it/s]

ValueError: Data correlation matrix is singular. Cannot run fisherz test. Please check your data.

In [14]:
corr =np.corrcoef(data, rowvar=False)
det = np.linalg.det(corr)
print("determinant: ", det)
cond = np.linalg.cond(corr)
print("condition number:", cond)

determinant:  0.0
condition number: 4.290235220181646e+17


In [16]:
#identify exactly duplicated columns
dup_cols = df_enc.T.duplicated()
print("Duplicated columns:", df_enc.columns[dup_cols].tolist())

#identify highly correlated pairs of columns
high_corr_pairs = []
threshold = 0.999999

for i in range(corr.shape[0]):
    for j in range(i+1, corr.shape[0]):
        if abs(corr[i, j]) > threshold:
            high_corr_pairs.append((df_enc.columns[i], df_enc.columns[j], corr[i, j]))

print("Near-perfectly correlated pairs:", high_corr_pairs[:20])


Duplicated columns: ['DrugTests_Cocaine_Positive_Missing', 'DrugTests_Meth_Positive_Missing', 'DrugTests_Other_Positive_Missing', 'Gender_F']
Near-perfectly correlated pairs: [('Gang_Affiliated_Missing', 'Gender_F', 1.0), ('Gang_Affiliated_Missing', 'Gender_M', -1.0), ('DrugTests_THC_Positive_Missing', 'DrugTests_Cocaine_Positive_Missing', 1.0), ('DrugTests_THC_Positive_Missing', 'DrugTests_Meth_Positive_Missing', 1.0), ('DrugTests_THC_Positive_Missing', 'DrugTests_Other_Positive_Missing', 1.0), ('DrugTests_Cocaine_Positive_Missing', 'DrugTests_Meth_Positive_Missing', 1.0), ('DrugTests_Cocaine_Positive_Missing', 'DrugTests_Other_Positive_Missing', 1.0), ('DrugTests_Meth_Positive_Missing', 'DrugTests_Other_Positive_Missing', 1.0), ('Gender_F', 'Gender_M', -1.0), ('Race_BLACK', 'Race_WHITE', -1.0)]
